In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import seaborn as sns
from collections import Counter
import json

import all_blocks
from merge_contracts import ContractsMerger, cost_columns_to_datetime
from generate_test import generate_preds
from get_distribution_utils import get_disrib_sums

pd.options.display.max_columns = 100
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Подгружаем и обрабатываем датку

In [2]:
pays_df1 = pd.read_excel("data/Счета на оплату 3800-2023.XLSX")
pays_df2 = pd.read_excel("data/Счета на оплату 4200-4000-3800-2024.XLSX")
pays_df3 = pd.read_excel("data/Счета на оплату 5400-2023.XLSX")
pays_df4 = pd.read_excel("data/Счета на оплату 5400-2024.XLSX")
pays_df5 = pd.read_excel("data/Счета на оплату 5500-2023.XLSX")

In [3]:
pays_df = pd.concat([pays_df1, pays_df2, pays_df3, pays_df4, pays_df5], axis=0)

In [4]:
merger_df = pd.read_excel("data/Связь договор - здания.XLSX")
main_costs_df = pd.read_excel("data/Основные средства.XLSX")
squares = pd.read_excel("data/Площади зданий.XLSX")
serv_codes = pd.read_excel("data/Коды услуг.XLSX")
main_costs_df["ФИЧА БЛЯ)"] = 1

In [7]:
merger = ContractsMerger(pays_df, merger_df, main_costs_df, squares, serv_codes)

E:\DIR_python_projects\ledaer_digital_transformation_24\NEW\merge_contracts.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.needed_pays_df["time"] = self.needed_pays_df["Год"].map(lambda x: datetime(year=x + 1, month=1, day=1))


In [15]:
import time

t = time.time()
res, NUMERIC_COLUMNS = merger.start_merging()
print(time.time() - t)

128.41562581062317


In [16]:
# res = pd.read_csv("res_datetimes.csv", index_col=0)

In [18]:
res.to_csv("res_datetimes.csv")

## Подгружаем граф, запускаем распределение

In [25]:
res = pd.read_csv("res_datetimes.csv", index_col=0)
res_by_prime = res.groupby("prime")

C:\Users\843E~1\AppData\Local\Temp/ipykernel_15752/3306859286.py:1: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  res = pd.read_csv("res_datetimes.csv", index_col=0)


In [33]:
raw_graph = json.load(open("graphs/graph_gleb.json", "r"))
all_blocks.init_graph(raw_graph)

In [39]:
unique_primes = res["prime"].unique()
distrib_sums = get_disrib_sums(res_by_prime, res["prime"].unique(), NUMERIC_COLUMNS)

100%|██████████| 4717/4717 [00:06<00:00, 688.59it/s]


In [37]:
preds = generate_preds(pays_df, serv_codes, res, distrib_sums)

100%|██████████| 124651/124651 [02:27<00:00, 845.12it/s] 


In [48]:
preds.to_csv("Результат_распределенные_счета.csv")